In [1]:
from pyod.models.copod import COPOD
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import os
import pickle

from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt 
%matplotlib inline

import sys
sys.path.append('..')
sys.path.append('../..')

from src.utils import *
from src.Components.data_processing import data_process

In [45]:
# Dodgers Dataset

file_path = '../../datasets/Dodgers/101-freeway-traffic.test.out'

columns = ['value', 'anomaly']

df = pd.read_csv(file_path, names=columns, header=None)

In [81]:
# NAB Tweet Data

file_path1 = '../../datasets/NAB/NAB_data_tweets_1.out'
file_path2 = '../../datasets/NAB/NAB_data_tweets_2.out'

columns = ['value', 'anomaly']

train_df = pd.read_csv(file_path1, names=columns, header=None)
test_df = pd.read_csv(file_path2, names=columns, header=None)

In [83]:
train_np = train_df[['value']][train_df['anomaly'] == 0]
train_np

,value
0,100.0
1,99.0
2,154.0
3,120.0
4,90.0
...,...
15896,44.0
15897,45.0
15898,48.0
15899,26.0


df = df[df['value'] >= 0]
df['anomaly'].value_counts()

In [46]:

X_train, X_test = train_test_split(df, test_size=0.3, shuffle=False)
train_x = X_train[['value']][X_train['anomaly']== 0]
train_y = X_train[['anomaly']].values.ravel()
test_data = X_test[['value']]
gtruth = X_test[['anomaly']]

In [47]:
model = COPOD(contamination=0.057)
model.fit(train_x)

COPOD(contamination=0.057, n_jobs=1)

In [84]:
model_tweet = COPOD(contamination=0.054)
model_tweet.fit(train_np)

COPOD(contamination=0.054, n_jobs=1)

### Evaluating the Models

In [48]:
anomaly_scores = model.decision_function(test_data)
y_pred = model.predict(test_data)

In [86]:
anomaly_scores_tweet = model.decision_function(test_df[['value']])
gtruth_tweet = test_df[['anomaly']]

In [90]:
thres_tweet = raw_thresholds(anomaly_scores_tweet, contamination=0.062)
thres_tweet = thres_tweet - 0.65
thres_tweet

3.2110158841136625

In [91]:
print(np.max(anomaly_scores_tweet))
print(np.min(anomaly_scores_tweet))
print(np.mean(anomaly_scores_tweet))

10.761746548158834
0.6702148674997177
1.9295738957749058


In [92]:
thres_np = [1 if x > thres_tweet else 0 for x in anomaly_scores_tweet]
thres_np

[0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [93]:
prec = precision_score(gtruth_tweet, thres_np)
recall = recall_score(gtruth_tweet, thres_np)
f1 = f1_score(gtruth_tweet, thres_np)

In [94]:
print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precision Score: 0.1763  Recall: 0.2081  f1_score: 0.1909


In [55]:
gtruth['anomaly'].value_counts()

anomaly
0    13437
1     1683
Name: count, dtype: int64

In [50]:
print(np.max(anomaly_scores))
print(np.min(anomaly_scores))
print(np.mean(anomaly_scores))

8.95482427187917
0.6702398867214648
1.3275750004000952


In [76]:
threshold = raw_thresholds(anomaly_scores, contamination=0.111)
threshold = threshold - 0.65
threshold

1.5831977880339423

In [77]:
thres_np = [1 if x > threshold else 0 for x in anomaly_scores]
thres_np

[0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 1,


In [78]:
prec = precision_score(gtruth, thres_np)
recall = recall_score(gtruth, thres_np)
f1 = f1_score(gtruth, thres_np)

In [79]:
print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precision Score: 0.2408  Recall: 0.5478  f1_score: 0.3345


In [38]:
prec = precision_score(gtruth, y_pred)
recall = recall_score(gtruth, y_pred)
f1 = f1_score(gtruth, y_pred)

In [80]:
file_name = f'../../saved_models/copod_dodgers_v2.sav'

pickle.dump(model, open(file_name, 'wb'))